In [0]:
# imports
from pyspark.sql.functions import col, current_timestamp, sum as spark_sum, when
from pyspark.sql.types import StringType

In [0]:
# OAuth configurações
def build_adls_options(storage_account_name, client_id, tenant_id, client_secret):
    return {
        f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net": "OAuth",
        f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
        f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net": client_id,
        f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net": client_secret,
        f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
    }

# leitura do csv da origem configurada no ADLS
def read_source_csv(spark, source_path, adls_options, csv_options=None):

    csv_options = csv_options or {}

    return (
        spark.read
        .format("csv")
        .options(**csv_options)
        .options(**adls_options)
        .load(source_path)
    )

In [0]:
# informações gerais do df
def get_dataframe_overview(df):
    """
    Retorna informações básicas sobre um DataFrame.
    Útil tanto para analysis quanto para ingestion.
    """

    return {
        "total_linhas": df.count(),
        "total_colunas": len(df.columns),
        "colunas": df.columns,
    }

# quantidade de valores nulos por coluna
def get_null_summary(df):

    return df.select([
        spark_sum(when(col(column_name).isNull(), 1).otherwise(0)).alias(column_name)
        for column_name in df.columns
    ])

In [0]:
# validações de colunas obrigatórias
def validate_required_columns(df, expected_columns):

    if not expected_columns:
        return {
            "validation_applied": False,
            "missing_columns": [],
            "unexpected_columns": df.columns,
            "message": "EXPECTED_COLUMNS não informado. Validação de colunas obrigatórias ignorada.",
        }

    missing_columns = [
        column_name
        for column_name in expected_columns
        if column_name not in df.columns
    ]

    unexpected_columns = [
        column_name
        for column_name in df.columns
        if column_name not in expected_columns
    ]

    if missing_columns:
        raise ValueError(f"Colunas obrigatórias ausentes: {missing_columns}")

    return {
        "validation_applied": True,
        "missing_columns": missing_columns,
        "unexpected_columns": unexpected_columns,
        "message": "Validação de colunas obrigatórias concluída com sucesso.",
    }


# validação se colunas candidatas a chave existem
def validate_key_columns(df, key_columns):

    if not key_columns:
        return {
            "validation_applied": False,
            "key_columns": [],
            "null_counts": {},
            "duplicated_rows": None,
            "message": "KEY_COLUMNS não informado. Validação de chave ignorada.",
        }

    missing_key_columns = [
        column_name
        for column_name in key_columns
        if column_name not in df.columns
    ]

    if missing_key_columns:
        raise ValueError(
            f"Colunas de chave ausentes no DataFrame: {missing_key_columns}"
        )

    null_counts = {}

    for column_name in key_columns:
        null_count = df.filter(col(column_name).isNull()).count()
        null_counts[column_name] = null_count

        if null_count > 0:
            raise ValueError(
                f"A coluna de chave {column_name} possui {null_count} registros nulos."
            )

    total_rows = df.count()
    distinct_key_rows = df.select(*key_columns).distinct().count()
    duplicated_rows = total_rows - distinct_key_rows

    if duplicated_rows > 0:
        raise ValueError(
            f"Foram encontrados {duplicated_rows} registros duplicados "
            f"considerando a chave {key_columns}."
        )

    return {
        "validation_applied": True,
        "key_columns": key_columns,
        "null_counts": null_counts,
        "duplicated_rows": duplicated_rows,
        "message": "Validação de chave concluída com sucesso.",
    }

# compara a quantidade de linhas entre dois dataframes de origem e destino
def compare_row_counts(source_df, target_df):

    source_count = source_df.count()
    target_count = target_df.count()

    if source_count != target_count:
        raise ValueError(
            f"Divergência entre origem e destino. "
            f"Origem: {source_count}. Destino: {target_count}."
        )

    return {
        "source_count": source_count,
        "target_count": target_count,
        "message": "Quantidade de registros validada com sucesso.",
    }

In [0]:
def add_load_timestamp(df, column_name="dt_carga"):
    """
    Adiciona uma coluna técnica com o timestamp da carga.

    Essa coluna representa o momento em que o dado foi processado
    pelo pipeline de ingestão.
    """

    return df.withColumn(column_name, current_timestamp())

In [0]:
def cast_all_columns_to_string(df):
    """
    Converte todas as colunas do DataFrame para StringType.

    Essa função é útil para cargas raw/bronze,
    onde queremos preservar os dados o mais próximo possível da origem,
    sem aplicar tipagem analítica como int, boolean, date ou timestamp.
    """

    for column_name in df.columns:
        df = df.withColumn(column_name, col(column_name).cast(StringType()))

    return df

In [0]:
# leitura da tabela no SQL Server
def read_sql_table(
    spark,
    sql_host,
    sql_database,
    sql_username,
    sql_password,
    table_name,
    sql_port="1433"
):
    return (
        spark.read
        .format("sqlserver")
        .option("host", sql_host)
        .option("port", sql_port)
        .option("database", sql_database)
        .option("dbtable", table_name)
        .option("user", sql_username)
        .option("password", sql_password)
        .load()
    )

# escrita da tabela no SQL Server
def write_sql_table(
    df,
    sql_host,
    sql_database,
    sql_username,
    sql_password,
    table_name,
    mode="overwrite",
    sql_port="1433"
):

    (
        df.write
        .format("sqlserver")
        .option("host", sql_host)
        .option("port", sql_port)
        .option("database", sql_database)
        .option("dbtable", table_name)
        .option("user", sql_username)
        .option("password", sql_password)
        .mode(mode)
        .save()
    )